# Rule-Based Engine & Hybrid Risk Scoring
## Combining Expert Rules with ML Anomaly Detection

---

### Why Rules + ML?
| Approach | Pros | Cons |
|----------|------|------|
| ML Only | จับ pattern ซับซ้อน | Black box, ยากอธิบาย |
| Rules Only | Interpretable, ควบคุมได้ | ไม่จับ pattern ใหม่ |
| **Hybrid** | **ได้ทั้ง 2 ข้อดี** | **ต้อง calibrate weights** |

### กฎที่สร้างจาก Insight ที่พบ
1. **High Login Attempts** — LoginAttempts ≥ 3 (risk premium +164%)
2. **Amount Spike** — Amount Z-Score > 2 (ยอดสูงผิดปกติเทียบกับบัญชี)
3. **High Amount-to-Balance** — ratio > 0.8 (ใช้เงินเกือบหมดยอดคงเหลือ)
4. **Device Sharing** — อุปกรณ์ถูกใช้กับ ≥ 5 บัญชี
5. **Rapid Transaction** — ทำธุรกรรมภายใน 1 ชม. จากครั้งก่อน
6. **Multi-Location Account** — บัญชีทำรายการใน 8+ เมือง
7. **High Daily Velocity** — ≥ 3 ธุรกรรมต่อวัน

---

## 1. Setup & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Load
df = pd.read_csv('bank_transactions.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed')

# Feature Engineering (same as notebook 02)
acct = df.groupby('AccountID').agg(
    acct_txn_count=('TransactionID','count'), acct_mean_amount=('TransactionAmount','mean'),
    acct_std_amount=('TransactionAmount','std'), acct_max_amount=('TransactionAmount','max'),
    acct_mean_balance=('AccountBalance','mean'), acct_mean_duration=('TransactionDuration','mean'),
    acct_mean_login=('LoginAttempts','mean'), acct_unique_devices=('DeviceID','nunique'),
    acct_unique_locations=('Location','nunique'), acct_unique_merchants=('MerchantID','nunique'),
    acct_unique_channels=('Channel','nunique'), acct_unique_ips=('IP Address','nunique')
).reset_index()
acct['acct_std_amount'] = acct['acct_std_amount'].fillna(0)
df = df.merge(acct, on='AccountID', how='left')

df['amount_zscore'] = (df['TransactionAmount'] - df['acct_mean_amount']) / df['acct_std_amount'].replace(0,1)
df['amount_to_balance_ratio'] = df['TransactionAmount'] / df['AccountBalance'].replace(0,1)
Q1, Q3 = df['TransactionAmount'].quantile(0.25), df['TransactionAmount'].quantile(0.75)
df['amount_outlier_flag'] = (df['TransactionAmount'] > Q3 + 1.5*(Q3-Q1)).astype(int)
df['high_login_flag'] = (df['LoginAttempts'] >= 3).astype(int)

dev = df.groupby('DeviceID')['AccountID'].nunique().reset_index()
dev.columns = ['DeviceID','device_shared_accounts']
df = df.merge(dev, on='DeviceID', how='left')
df['device_shared_flag'] = (df['device_shared_accounts'] > 1).astype(int)

ip_cnt = df.groupby('IP Address')['AccountID'].nunique().reset_index()
ip_cnt.columns = ['IP Address','ip_shared_accounts']
df = df.merge(ip_cnt, on='IP Address', how='left')

mf = df.groupby(['AccountID','MerchantID']).size().reset_index(name='merchant_visit_count')
df = df.merge(mf, on=['AccountID','MerchantID'], how='left')
df['is_new_merchant'] = (df['merchant_visit_count'] == 1).astype(int)

df['hour'] = df['TransactionDate'].dt.hour
df['day_of_week'] = df['TransactionDate'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['txn_date'] = df['TransactionDate'].dt.date
dv = df.groupby(['AccountID','txn_date']).size().reset_index(name='daily_txn_count')
df = df.merge(dv, on=['AccountID','txn_date'], how='left')
df = df.sort_values(['AccountID','TransactionDate'])
df['time_since_last_txn'] = df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds()/3600
df['time_since_last_txn'] = df['time_since_last_txn'].fillna(-1)
df['rapid_txn_flag'] = ((df['time_since_last_txn']>=0)&(df['time_since_last_txn']<1)).astype(int)
df['duration_deviation'] = abs(df['TransactionDuration'] - df['acct_mean_duration'])
df['amount_to_max_ratio'] = df['TransactionAmount'] / df['acct_max_amount'].replace(0,1)
df.drop(columns=['txn_date'], inplace=True)

print(f'Data ready: {df.shape[0]:,} rows x {df.shape[1]} columns')

---
## 2. Define Fraud Rules

แต่ละกฎมี **severity weight** ตาม risk premium ที่พบจาก Model Evaluation

In [ ]:
# ── Define Rules with Severity Weights ──
rules = {
    'R1_high_login': {
        'condition': df['LoginAttempts'] >= 3,
        'weight': 3.0,
        'severity': 'Critical',
        'description': 'Login Attempts ≥ 3 (brute force indicator)'
    },
    'R2_amount_spike': {
        'condition': df['amount_zscore'] > 2,
        'weight': 2.5,
        'severity': 'High',
        'description': 'Amount Z-Score > 2 (unusually high for this account)'
    },
    'R3_high_ratio': {
        'condition': df['amount_to_balance_ratio'] > 0.8,
        'weight': 2.0,
        'severity': 'High',
        'description': 'Amount-to-Balance ratio > 0.8 (near full balance usage)'
    },
    'R4_device_sharing': {
        'condition': df['device_shared_accounts'] >= 5,
        'weight': 2.5,
        'severity': 'Critical',
        'description': 'Device shared across ≥ 5 accounts'
    },
    'R5_rapid_txn': {
        'condition': df['rapid_txn_flag'] == 1,
        'weight': 1.5,
        'severity': 'Medium',
        'description': 'Transaction within 1 hour of previous'
    },
    'R6_multi_location': {
        'condition': df['acct_unique_locations'] >= 8,
        'weight': 2.0,
        'severity': 'High',
        'description': 'Account active in ≥ 8 cities'
    },
    'R7_high_velocity': {
        'condition': df['daily_txn_count'] >= 3,
        'weight': 1.5,
        'severity': 'Medium',
        'description': '≥ 3 transactions in a single day'
    }
}

# Apply rules
for rule_name, rule in rules.items():
    df[rule_name] = rule['condition'].astype(int)

# Summary
print('Rule-Based Engine — 7 Rules Defined:\n')
print(f'{"Rule":>20s} {"Severity":>10s} {"Weight":>8s} {"Triggered":>10s} {"Pct":>8s}')
print('-' * 62)
rule_names = list(rules.keys())
for rn in rule_names:
    r = rules[rn]
    n = df[rn].sum()
    print(f'{rn:>20s} {r["severity"]:>10s} {r["weight"]:>8.1f} {n:>10,} {n/len(df)*100:>7.2f}%')

print(f'\nTotal transactions: {len(df):,}')

In [ ]:
# ── Rule Score Calculation ──
max_possible = sum(r['weight'] for r in rules.values())

df['rule_score_raw'] = sum(df[rn] * rules[rn]['weight'] for rn in rule_names)
df['rule_score'] = df['rule_score_raw'] / max_possible  # Normalize to 0-1
df['rules_triggered'] = sum(df[rn] for rn in rule_names)

print(f'Max possible raw score: {max_possible}')
print(f'\nRule Score Distribution:')
print(df['rule_score'].describe().round(4))
print(f'\nRules triggered per transaction:')
for n in range(int(df['rules_triggered'].max())+1):
    c = (df['rules_triggered']==n).sum()
    if c > 0:
        print(f'  {n} rules: {c:>6,} ({c/len(df)*100:.1f}%)')

In [ ]:
# ── Visualization: Rule triggers ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Rule trigger counts
trigger_counts = pd.Series({rn: df[rn].sum() for rn in rule_names}).sort_values()
colors = ['#ef4444' if rules[rn]['severity']=='Critical' else '#f59e0b' if rules[rn]['severity']=='High' else '#3b82f6'
          for rn in trigger_counts.index]
trigger_counts.plot(kind='barh', ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Rule Trigger Frequency')
axes[0].set_xlabel('Transactions Triggered')
for i, (rn, v) in enumerate(trigger_counts.items()):
    axes[0].text(v+100, i, f'{v:,}', va='center', fontweight='bold', fontsize=9)

# 2. Rules triggered distribution
rule_dist = df['rules_triggered'].value_counts().sort_index()
bar_colors = ['#22c55e','#3b82f6','#f59e0b','#ef4444','#7f1d1d','#4a0404','#000']
rule_dist.plot(kind='bar', ax=axes[1], color=bar_colors[:len(rule_dist)], edgecolor='black', linewidth=0.5)
axes[1].set_title('Rules Triggered per Transaction')
axes[1].set_xlabel('Number of Rules')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# 3. Rule score distribution
axes[2].hist(df['rule_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[2].set_title('Rule Score Distribution')
axes[2].set_xlabel('Rule Score (0-1)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 3. ML Scores (reproduce)

รัน 3 models เหมือน Notebook 02 เพื่อนำมารวมกับ Rule Score

In [ ]:
# ── Run ML Models ──
features = [
    'TransactionAmount','TransactionDuration','LoginAttempts','AccountBalance',
    'amount_zscore','amount_to_balance_ratio','amount_to_max_ratio',
    'duration_deviation','high_login_flag','amount_outlier_flag',
    'device_shared_accounts','device_shared_flag','ip_shared_accounts','is_new_merchant',
    'acct_txn_count','acct_std_amount','acct_unique_devices','acct_unique_locations',
    'daily_txn_count','rapid_txn_flag','is_weekend','CustomerAge'
]
X = df[features].replace([np.inf,-np.inf], np.nan).fillna(0)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features, index=X.index)
X_pca = PCA(n_components=5, random_state=42).fit_transform(X_scaled)

# Models
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
df['iso_anomaly'] = (iso.fit_predict(X_scaled)==-1).astype(int)
df['iso_score'] = iso.decision_function(X_scaled)

db = DBSCAN(eps=2.5, min_samples=10, n_jobs=-1)
df['db_anomaly'] = (db.fit_predict(X_pca)==-1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
df['lof_anomaly'] = (lof.fit_predict(X_scaled)==-1).astype(int)
df['lof_score'] = -lof.negative_outlier_factor_

# ML composite score
df['iso_norm'] = MinMaxScaler().fit_transform(-df[['iso_score']])
df['lof_norm'] = MinMaxScaler().fit_transform(df[['lof_score']])
df['ml_score'] = 0.45*df['iso_norm'] + 0.35*df['lof_norm'] + 0.20*df['db_anomaly'].astype(float)

print('ML Models fitted:')
print(f'  ISO anomalies: {df["iso_anomaly"].sum():,}')
print(f'  DBSCAN noise:  {df["db_anomaly"].sum():,}')
print(f'  LOF anomalies: {df["lof_anomaly"].sum():,}')
print(f'  ML Score range: [{df["ml_score"].min():.4f}, {df["ml_score"].max():.4f}]')

---
## 4. Hybrid Risk Score

**Hybrid = α × ML Score + (1-α) × Rule Score**

α = 0.5 (equal weight) — ปรับได้ตามความต้องการของ business

In [ ]:
# ── Hybrid Risk Score ──
alpha = 0.5  # ML weight
df['hybrid_score'] = alpha * df['ml_score'] + (1 - alpha) * df['rule_score']

# Risk levels
df['hybrid_level'] = pd.cut(df['hybrid_score'],
    bins=[0, 0.1, 0.25, 0.45, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical'],
    include_lowest=True
)

print('Hybrid Risk Score Distribution:')
print(df['hybrid_score'].describe().round(4))
print(f'\nHybrid Risk Levels:')
for level in ['Low', 'Medium', 'High', 'Critical']:
    c = (df['hybrid_level'] == level).sum()
    print(f'  {level:10s}: {c:>6,} ({c/len(df)*100:.1f}%)')

# Compare ML-only vs Rule-only vs Hybrid
print(f'\nScore Comparison:')
print(f'  ML Score mean:     {df["ml_score"].mean():.4f}')
print(f'  Rule Score mean:   {df["rule_score"].mean():.4f}')
print(f'  Hybrid Score mean: {df["hybrid_score"].mean():.4f}')

In [ ]:
# ── Visualization: Hybrid Score ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ML vs Rule scatter
scatter = axes[0,0].scatter(df['ml_score'], df['rule_score'], c=df['hybrid_score'],
                            cmap='YlOrRd', alpha=0.2, s=5, rasterized=True)
axes[0,0].set_title('ML Score vs Rule Score\n(color = Hybrid Score)')
axes[0,0].set_xlabel('ML Score'); axes[0,0].set_ylabel('Rule Score')
plt.colorbar(scatter, ax=axes[0,0], label='Hybrid Score')

# 2. Hybrid score distribution
axes[0,1].hist(df['hybrid_score'], bins=60, color='steelblue', edgecolor='white', alpha=0.7)
for t, l, c in [(0.1,'Low|Med','#22c55e'),(0.25,'Med|High','#f59e0b'),(0.45,'High|Crit','#ef4444')]:
    axes[0,1].axvline(t, color=c, linestyle='--', linewidth=2, label=f'{l} ({t})')
axes[0,1].set_title('Hybrid Risk Score Distribution')
axes[0,1].set_xlabel('Hybrid Score'); axes[0,1].legend(fontsize=8)

# 3. Risk level counts
level_counts = df['hybrid_level'].value_counts().reindex(['Low','Medium','High','Critical'])
colors_bar = ['#22c55e','#f59e0b','#ef4444','#7f1d1d']
bars = axes[1,0].bar(level_counts.index, level_counts.values, color=colors_bar, edgecolor='black', linewidth=0.5)
axes[1,0].set_title('Hybrid Risk Level Distribution')
for bar, v in zip(bars, level_counts.values):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, v+200,
                   f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=10)

# 4. Hybrid vs Amount
axes[1,1].scatter(df['TransactionAmount'], df['hybrid_score'],
                  c=df['rules_triggered'], cmap='RdYlGn_r', alpha=0.15, s=5, rasterized=True)
axes[1,1].axhline(0.25, color='#f59e0b', linestyle='--', alpha=0.5)
axes[1,1].axhline(0.45, color='#ef4444', linestyle='--', alpha=0.5)
axes[1,1].set_title('Hybrid Score vs Transaction Amount')
axes[1,1].set_xlabel('Amount ($)'); axes[1,1].set_ylabel('Hybrid Score')

plt.tight_layout()
plt.show()

---
## 5. Rule Effectiveness Analysis

In [ ]:
# ── Which rules are most effective at catching ML anomalies? ──
ml_anomaly = (df['iso_anomaly'] + df['db_anomaly'] + df['lof_anomaly']) >= 2

print('Rule Effectiveness — Catching ML Consensus Anomalies (2+ models):\n')
print(f'{"Rule":>20s} {"Triggered":>10s} {"Catches ML":>12s} {"Precision":>10s} {"Recall":>10s}')
print('-' * 68)

n_ml_anom = ml_anomaly.sum()
for rn in rule_names:
    triggered = df[rn].sum()
    catches = (df[rn] & ml_anomaly).sum()
    precision = catches / max(triggered, 1) * 100
    recall = catches / max(n_ml_anom, 1) * 100
    print(f'{rn:>20s} {triggered:>10,} {catches:>12,} {precision:>9.1f}% {recall:>9.1f}%')

# Combined rules
any_rule = df[rule_names].any(axis=1)
catches_any = (any_rule & ml_anomaly).sum()
print(f'{"ANY RULE":>20s} {any_rule.sum():>10,} {catches_any:>12,} {catches_any/max(any_rule.sum(),1)*100:>9.1f}% {catches_any/max(n_ml_anom,1)*100:>9.1f}%')

In [ ]:
# ── Rule Co-occurrence Heatmap ──
rule_matrix = df[rule_names].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(rule_matrix, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=1, linecolor='white', vmin=0, vmax=1,
            xticklabels=[rn.replace('_',' ') for rn in rule_names],
            yticklabels=[rn.replace('_',' ') for rn in rule_names])
ax.set_title('Rule Co-occurrence Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Export Hybrid Results

In [ ]:
# ── Export ──
export_cols = [
    'TransactionID','AccountID','TransactionAmount','TransactionDate','TransactionType',
    'Location','DeviceID','Channel','CustomerAge','CustomerOccupation',
    'TransactionDuration','LoginAttempts','AccountBalance',
    # Rules
    'R1_high_login','R2_amount_spike','R3_high_ratio','R4_device_sharing',
    'R5_rapid_txn','R6_multi_location','R7_high_velocity',
    'rules_triggered','rule_score',
    # ML
    'iso_anomaly','db_anomaly','lof_anomaly','ml_score',
    # Hybrid
    'hybrid_score','hybrid_level'
]

df[export_cols].to_csv('bank_transactions_hybrid_scores.csv', index=False)
print(f'Exported: bank_transactions_hybrid_scores.csv')
print(f'Shape: {df[export_cols].shape}')
print(f'\nHybrid Risk Summary:')
for level in ['Low','Medium','High','Critical']:
    c = (df['hybrid_level']==level).sum()
    print(f'  {level:10s}: {c:>6,} ({c/len(df)*100:.1f}%)')